# External validation on IDRiD: Kaggle notebook (one track per notebook)

Scores the **frozen** winner and baseline on IDRiD, a dataset the models have never seen. No retraining, no fine-tuning, no threshold tuning.
Each run is scored once per dataset (the code refuses a second scoring). Use a **separate copy of this notebook for each track**.

## Inputs to add (right panel -> Add Input)

| # | Input | Grading notebook | Segmentation notebook |
|---|---|---|---|
| 1 | **IDRiD dataset**: Datasets -> `dankok/diabetic-retinopathy-image-dataset` (has both the grading and the segmentation part) | yes | yes |
| 2 | **Your results notebook output** (Your Work -> Notebook Output): the latest version that holds the final runs, their `best.pt` and `test_metrics.json` | **DDR-grade2** | **DDR-seg3** |
| 3 | (optional) **APTOS 2019**: Competitions -> `aptos2019-blindness-detection` (join the competition and accept the rules first) | only if `RUN_APTOS = True` | no |

Attach **only one** results notebook output per notebook (the restore step takes the first match). You do **not** need the DDR datasets here.

## Session options
**Internet = On** (needed for `git clone` and `pip`) and **Accelerator = GPU** (T4 or P100; segmentation evaluates full 1024 px images).

## Then
1. Edit the settings cell below (`TRACK`, and `RUN_APTOS` if you added APTOS).
2. **Run All.** The sanity cell stops with a clear message if a model is missing.
3. When it finishes, **Save Version** and download `outputs/external_*.csv` and the `idrid_metrics.json` files from the Output tab.

In [ ]:
# ---------------------------- SETTINGS (edit me) ----------------------------
TRACK = "grading"                  # "grading" or "segmentation"
RUN_APTOS = False                  # grading only: also evaluate on APTOS 2019 (needs input #3)
SEEDS = (42, 43, 44)               # the seeds you trained for the winner and the baseline
WINNER = None                      # None = the frozen winner (g4_focal / s1_weighted_bce); set a name only if you froze another one
BRANCH = "claude/sweet-sagan-8khcuq"   # repo branch (change to "main" once merged)
REPO_URL = "https://github.com/kazimab1/DDR-DR"

In [ ]:
# ---------------------------- SETUP ----------------------------
import glob, os, subprocess, sys
try:
    from IPython.display import display
except ImportError:                                  # only when run outside a notebook
    display = print
assert TRACK in ("grading", "segmentation"), "TRACK must be 'grading' or 'segmentation'"
WINNER = WINNER or {"grading": "g4_focal", "segmentation": "s1_weighted_bce"}[TRACK]
BASELINE = {"grading": "g0_baseline", "segmentation": "s0_baseline"}[TRACK]

def sh(cmd):
    """Run a shell command, stream its output (progress-bar noise removed), stop on error."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "%|" not in line:
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed: {cmd}")

REPO = "/kaggle/working/DDR-DR"
if not os.path.isdir(REPO):
    sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO}")
os.chdir(REPO)
sh("pip install -q timm segmentation-models-pytorch")   # torch & co. are preinstalled on Kaggle; do not reinstall them
import torch
print("GPU available:", torch.cuda.is_available(), "| track:", TRACK, "| winner:", WINNER, "| baseline:", BASELINE)
if not torch.cuda.is_available():
    print("WARNING: no GPU. Segmentation at 1024 px will be slow on CPU. Set Accelerator to GPU in Session options.")

In [ ]:
# ---------------------------- RESTORE the trained runs from your results notebook output ----------------------------
def find_previous():
    for pattern in ("/kaggle/input/*/DDR-DR", "/kaggle/input/*/*/DDR-DR", "/kaggle/input/*/*/*/DDR-DR", "/kaggle/input/*/*/*/*/DDR-DR"):
        for d in glob.glob(pattern):
            if os.path.isdir(os.path.join(d, "outputs", "runs")):
                return d
    return None

prev = find_previous()
if prev is None:
    raise FileNotFoundError("No results notebook output found under /kaggle/input. Add it via Add Input -> Notebook Output "
                            "(DDR-grade2 for grading, DDR-seg3 for segmentation) and run again.")
print("Restoring runs from", prev)
os.makedirs("outputs/runs", exist_ok=True)
sh(f"cp -rn '{prev}/outputs/runs/.' outputs/runs/")      # -n: never overwrite

# every frozen model must be present before anything slow starts
suffix = lambda s: "" if s == 42 else f"_s{s}"
missing = [f"outputs/runs/{m}{suffix(s)}/best.pt" for m in (WINNER, BASELINE) for s in SEEDS
           if not os.path.exists(f"outputs/runs/{m}{suffix(s)}/best.pt")]
if missing:
    raise FileNotFoundError("Missing trained models (is the right results notebook attached, and is it the latest version?):\n  " + "\n  ".join(missing))
no_test = [f"{m}{suffix(s)}" for m in (WINNER, BASELINE) for s in SEEDS
           if not os.path.exists(f"outputs/runs/{m}{suffix(s)}/test_metrics.json")]
print("All", 2 * len(SEEDS), "models found.", ("(No DDR test_metrics.json for: %s; the DDR-vs-external comparison will show nan for them.)" % no_test) if no_test else "DDR test metrics found too.")

In [ ]:
# ---------------------------- LOCATE IDRiD ----------------------------
def find_idrid():
    for depth in ("*", "*/*", "*/*/*", "*/*/*/*"):
        for d in sorted(glob.glob(f"/kaggle/input/{depth}")):
            if os.path.isdir(os.path.join(d, "Disease_Grading")) and os.path.isdir(os.path.join(d, "Segmentation")):
                return d
    raise FileNotFoundError("IDRiD not found under /kaggle/input. Add dankok/diabetic-retinopathy-image-dataset via Add Input. Inputs: "
                            + str(os.listdir("/kaggle/input")))
IDRID = find_idrid()
print("IDRiD folder:", IDRID)
sh(f"ls '{IDRID}' '{IDRID}/Disease_Grading' '{IDRID}/Segmentation'")

In [ ]:
# ---------------------------- RUN: IDRiD ----------------------------
if TRACK == "grading":
    base = f"{IDRID}/Disease_Grading"
    labels = sorted(glob.glob(f"{base}/Groundtruths/*.csv"))
    print("label files:", labels)
    assert len(labels) == 2, "expected the training-labels and testing-labels CSVs in Disease_Grading/Groundtruths"
    sh(f"python scripts/external_eval.py --track grading --name idrid --src '{base}' "
       f"--labels {' '.join(repr(l) for l in labels)} --winner {WINNER} --baseline {BASELINE} --seeds {' '.join(map(str, SEEDS))}")
else:
    sh(f"python scripts/external_eval.py --track segmentation --name idrid --src '{IDRID}/Segmentation' "
       f"--winner {WINNER} --baseline {BASELINE} --seeds {' '.join(map(str, SEEDS))}")

In [ ]:
# ---------------------------- RUN (optional): APTOS 2019, grading only ----------------------------
if RUN_APTOS:
    assert TRACK == "grading", "APTOS has grades only: use it in the grading notebook"
    aptos = next((d for d in glob.glob("/kaggle/input/*") if "aptos" in os.path.basename(d).lower()), None)
    assert aptos, "APTOS input not found: add the competition aptos2019-blindness-detection via Add Input"
    sh(f"python scripts/external_eval.py --track grading --name aptos --src '{aptos}' --labels '{aptos}/train.csv' "
       f"--winner {WINNER} --baseline {BASELINE} --seeds {' '.join(map(str, SEEDS))}")
    # --labels train.csv matters: the competition folder also has a sample_submission.csv with dummy all-zero grades
else:
    print("RUN_APTOS is False: skipped.")

In [ ]:
# ---------------------------- RESULT TABLES ----------------------------
import pandas as pd
for f in sorted(glob.glob("outputs/external_*.csv")):
    df = pd.read_csv(f)
    name = [c for c in df.columns if c.endswith("_mean") and c != "ddr_test_mean"][0][:-5]
    print("\n", f)
    show = df[["model", "metric", "ddr_test_mean", f"{name}_mean", f"{name}_std", "n_seeds"]].copy()
    show["change"] = show[f"{name}_mean"] - show["ddr_test_mean"]
    display(show.round(3))

## After this notebook

1. **Save Version -> Save & Run All (Commit)** (or Quick Save), then download from the Output tab: `outputs/external_*.csv` and the `idrid_metrics.json` (and `aptos_metrics.json`) of every run.
2. Send the printed tables (or the CSVs) so they can be added to the README, then do the other track in a second copy of this notebook.
3. How to read the numbers: see `TEST_EXTERNAL.md`. A large drop from DDR to external is a finding to discuss, not something to hide, and nothing should be tuned to recover it.